# Assignment 2
This project uses binary sentiment classifier for movie reviews, 0 = negative and 1 = positive. It utilizes a frozen pretrained model with a trained linear classification head.

## results
Public test accuracy achived: 0.7650, balanced accuracy = 0.7650, confusion matrix: [[141, 59], [35, 165]]

## Data inspection
It was found that there exsisted 3 primmary constraints that guide the design. The training set was skewed 75/25, while the test set was balanced 50/50. 210 of 240 reviews exceed the encoder's 512 token limit. There was no duplicate text between train and test sets so test scores avoid showing leakage.

In [1]:
import pandas as pd
import numpy as np
import sklearn
import torch
print("ok")

ok


In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/public_test.csv")

print(train.shape)
print(test.shape)

print(train["label"].value_counts())
print(test["label"].value_counts())
    
print(train.groupby(["label", "label_name"]).size())

(240, 5)
(400, 5)
label
1    180
0     60
Name: count, dtype: int64
label
1    200
0    200
Name: count, dtype: int64
label  label_name
0      negative       60
1      positive      180
dtype: int64


In [3]:
train.head()

,id,text,label,label_name,source_file
0,pos_cv230_7428,"well , i'll admit when i first heard about thi...",1,positive,pos/cv230_7428.txt
1,pos_cv853_29233,my summer was recently saved by two very diffe...,1,positive,pos/cv853_29233.txt
2,pos_cv771_28665,in october of 1962 the united states found its...,1,positive,pos/cv771_28665.txt
3,pos_cv449_8785,this is a good year if you want plenty of sci-...,1,positive,pos/cv449_8785.txt
4,pos_cv130_17083,"while watching wes anderson's rushmore , it ma...",1,positive,pos/cv130_17083.txt


In [4]:
print(len(set(train["source_file"]) & set(test["source_file"])))
print(train["text"].duplicated().sum())
print(train["text"].str.split().str.len().describe())

0
0
count     240.000000
mean      787.708333
std       364.790842
min        17.000000
25%       555.250000
50%       732.500000
75%       939.750000
max      2570.000000
Name: text, dtype: float64


# Eval Metric
All models are scored with a shared function that uses predictions rather than a model. Its to ensure that everything can be measured the same.

The model uses a balanced accuracy which averages per class accuracies, resulting in a 0.5 floor valure regaurdless of class proportion.

In [5]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

def evaluate_model(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    print(f"Accuracy: {acc:.4f}")
    print(f"Balanced Accuracy: {bal_acc:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    return acc, bal_acc, cm

In [6]:
train_baseline = np.ones(len(train), dtype=int)
test_baseline = np.ones(len(test), dtype=int)


print("All positive baseline on train set:")
evaluate_model(train["label"], train_baseline)

print("\nAll positive baseline on test set:")
evaluate_model(test["label"], test_baseline)

All positive baseline on train set:
Accuracy: 0.7500
Balanced Accuracy: 0.5000
Confusion Matrix:
[[  0  60]
 [  0 180]]

All positive baseline on test set:
Accuracy: 0.5000
Balanced Accuracy: 0.5000
Confusion Matrix:
[[  0 200]
 [  0 200]]


(0.5,
 0.5,
 array([[  0, 200],
        [  0, 200]]))

In [7]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

token_counts = pd.Series([len(tokenizer(t)["input_ids"]) for t in train["text"]])

print(token_counts.describe())
print(f"\n Reviews with over 512 tokens: {(token_counts > 512).sum()} out of {len(token_counts)}")


c:\Users\dleon\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (784 > 512). Running this sequence through the model will result in indexing errors


count     240.000000
mean      896.933333
std       409.528022
min        27.000000
25%       635.500000
50%       852.500000
75%      1078.000000
max      2848.000000
dtype: float64

 Reviews with over 512 tokens: 210 out of 240


In [8]:
def chunk_text(text, max_length=512, overlap=0):
    token_id = tokenizer(text, add_special_tokens=False)["input_ids"]
    body_size = max_length - 2  # account for [CLS] and [SEP] tokens
    step = body_size - overlap
    chunks = []
    for i in range(0, len(token_id), step):
        body = token_id[i:i + body_size]
        chunks.append([tokenizer.cls_token_id] + body + [tokenizer.sep_token_id])
        if i + body_size >= len(token_id):
            break
    return chunks


longest = train["text"].iloc[token_counts.idxmax()]
c = chunk_text(longest)
print(f"chunks: {len(c)}, sizes: {[len(x) for x in c]}")
print(f"total body tokens: {sum(len(x) - 2 for x in c)} (expected {token_counts.max() - 2})")

c2 = chunk_text(longest, overlap=100)
print(f"overlap=100 -> chunks: {len(c2)}, sizes: {[len(x) for x in c2]}")

chunks: 6, sizes: [512, 512, 512, 512, 512, 298]
total body tokens: 2846 (expected 2846)
overlap=100 -> chunks: 7, sizes: [512, 512, 512, 512, 512, 512, 388]


In [9]:
#Frozen econder
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained("distilbert-base-uncased")
model.eval()

print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4252.52it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


parameters: 66,362,880


In [10]:
@torch.no_grad()
def get_embeddings(text, overlap=0):
    chunks =  chunk_text(text, overlap=overlap)
    vectors = []
    for c in chunks:
        input_ids = torch.tensor([c])
        outputs = model(input_ids)
        vectors.append(outputs.last_hidden_state[:, 0, :].squeeze(0))
    return torch.stack(vectors).mean(dim=0).numpy()

v = get_embeddings(train["text"].iloc[0])
print(v.shape)

v_long = get_embeddings(longest)
print(v_long.shape)

(768,)
(768,)


In [11]:
from tqdm.auto import tqdm
import numpy as np

x_train = np.array([get_embeddings(t) for t in tqdm(train["text"], desc="train")])
x_test = np.array([get_embeddings(t) for t in tqdm(test["text"], desc="test")])

y_train = train["label"].values
y_test = test["label"].values


print(x_train.shape, x_test.shape)

np.savez("embeddings_overlap_0.npz", x_train=x_train, y_train=y_train, x_test=x_test, y_test=y_test)


test: 100%|██████████| 400/400 [01:27<00:00,  4.58it/s]

(240, 768) (400, 768)


## Dealing with imbalance and small traninig set

To deal with the imbalance weighted cross entropy with weights n_total / (2 * n_class). Which gives a heaveier weight to negative examples.

Class weight|CV balanced accuracy: None | 0.7306, balanced | 0.7861

The decision threshold had alreafy been picked by CV, it came out around 0.51, this indicated that weighted loss had already been calibrated a boundary.

To deal with potential unseen tokens The use of DistilBERT's subword tokenizer decomposes unseen words into known units, this helps prevent out of vocabulary issues.

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(5, shuffle=True, random_state=0)

for cw in [None, "balanced"]:
    clf = LogisticRegression(max_iter=2000, class_weight=cw)
    scores = cross_val_score(clf, x_train, y_train, cv=cv, scoring="balanced_accuracy")
    print(f"class_weight={str(cw):<10} CV balanced accuracy: {scores.mean():.4f} (folds: {scores.round(3)})")

class_weight=None       CV balanced accuracy: 0.7306 (folds: [0.653 0.736 0.736 0.819 0.708])
class_weight=balanced   CV balanced accuracy: 0.7861 (folds: [0.736 0.847 0.847 0.847 0.653])


## Overfitting
 C | Train | CV | Gap |
|---|---|---|---|
| 0.001 | 0.7833 | 0.6944 | 0.089 |
| 0.01 | 0.8139 | 0.7083 | 0.106 |
| 0.1 | 0.8972 | 0.7556 | 0.142 |
| **1** | **0.9722** | **0.7861** | **0.186** |
| 10 | 1.0000 | 0.7694 | 0.231 |

Failure appears at c = 0.001 which shows underfitting and c=10 had the better accuracy with 0.7694 generalization. Even with this we only achive 0.7694 because of the limit on parameters.

Opted for a 5 fold CV instead of a held out split, the reason being a 20% split leaves 12 negatives, which is too few.

Chunk overlap of 100 tokens was tested but it was determined that non-overlaping chunking would be fine and simpler to implement. Only because the differences fall within one fold std.

In [13]:
for C in [0.001, 0.01, 0.1, 1, 10]:
    clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=C)
    scores = cross_val_score(clf, x_train, y_train, cv=cv, scoring="balanced_accuracy")
    clf.fit(x_train, y_train)
    train_score = balanced_accuracy_score(y_train, clf.predict(x_train))
    print(f"C={C:<7} train: {train_score:.4f}  CV: {scores.mean():.4f} (±{scores.std():.3f})")    

C=0.001   train: 0.7833  CV: 0.6944 (±0.091)
C=0.01    train: 0.8139  CV: 0.7083 (±0.090)
C=0.1     train: 0.8972  CV: 0.7556 (±0.068)
C=1       train: 0.9722  CV: 0.7861 (±0.079)
C=10      train: 1.0000  CV: 0.7694 (±0.064)


In [14]:
x_train_ov = np.array([get_embeddings(t, overlap=100) for t in tqdm(train["text"], desc="train ov100")])
x_test_ov = np.array([get_embeddings(t, overlap=100) for t in tqdm(test["text"], desc="test ov100")])

print(x_train_ov.shape, x_test_ov.shape)

np.savez("embeddings_overlap_100.npz", x_train=x_train_ov, y_train=y_train, x_test=x_test_ov, y_test=y_test)

test ov100: 100%|██████████| 400/400 [01:40<00:00,  4.00it/s]

(240, 768) (400, 768)


In [15]:
for C in [0.1, 1, 10]:
    clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=C)
    scores = cross_val_score(clf, x_train_ov, y_train, cv=cv, scoring="balanced_accuracy")
    print(f"overlap=100  C={C:<5} CV: {scores.mean():.4f} (±{scores.std():.3f})")

overlap=100  C=0.1   CV: 0.7528 (±0.064)
overlap=100  C=1     CV: 0.7806 (±0.075)
overlap=100  C=10    CV: 0.7944 (±0.070)


# Model structure
Frozen encoder - distilbert-base-uncased, roughly 66 million parameters, none updated. Each review was split into non-overlapping 512 token chunks, each chunk passed through the encoder and the final layer CLS vector was taken. The vectors are then averaged into a 768 dimmensional representation for each review. It was shown that 87.5% of reviews exceed 512 tokens, longest needed 512 tokens. To verfiy this we compared total body toeken vs the original count.

Head - One nn.Linear(768, 2) layer, 1,538 parameters, trained on the 240 reviews. Softmax gives the positive class probability.

training Techniques - Setting|Value|Reason = 
Optimizer|Adam|Adaptive learning rate, 
Learning rate|1e-3| Head trained from scratch, not fine-tuned, Batch size | 32 | approx 8 steps per epoch over 240 examples, 
Epochs|100|Convex objective, converges reliably, Loss|Weighted cross-entropy|  Compensates 180/60 imbalance, Weight decay | 1e-2|L2 regularization, Seed|0|Reproducibility




In [16]:
import torch.nn as nn

def train_head(X, y, epochs=100, lr=1e-3, batch_size=32, seed=0):
    torch.manual_seed(seed)
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.long)
    head = nn.Linear(768, 2)
    counts = torch.bincount(yt).float()
    weights = counts.sum() / (2 * counts)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    opt = torch.optim.Adam(head.parameters(), lr=lr, weight_decay=1e-2)
    n = len(yt)
    for epoch in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            loss = loss_fn(head(Xt[idx]), yt[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    return head

@torch.no_grad()
def head_probs(head, X):
    logits = head(torch.tensor(X, dtype=torch.float32))
    return torch.softmax(logits, dim=1)[:, 1].numpy()

fold_scores = []
for tr_idx, va_idx in cv.split(x_train, y_train):
    h = train_head(x_train[tr_idx], y_train[tr_idx])
    preds = (head_probs(h, x_train[va_idx]) >= 0.5).astype(int)
    fold_scores.append(balanced_accuracy_score(y_train[va_idx], preds))
fold_scores = np.array(fold_scores)
print(f"PyTorch head CV: {fold_scores.mean():.4f} (±{fold_scores.std():.3f})")

PyTorch head CV: 0.7778 (±0.071)


In [17]:
oof_probs = np.zeros(len(y_train))
for tr_idx, va_idx in cv.split(x_train, y_train):
    h = train_head(x_train[tr_idx], y_train[tr_idx])
    oof_probs[va_idx] = head_probs(h, x_train[va_idx])

best_t, best_s = 0.5, 0
for t in np.arange(0.20, 0.81, 0.01):
    s = balanced_accuracy_score(y_train, (oof_probs >= t).astype(int))
    if s > best_s:
        best_s, best_t = s, t
print(f"best threshold={best_t:.2f}  CV balanced acc={best_s:.4f}")

best threshold=0.51  CV balanced acc=0.7889


## Eval results
Final head trained on 240 examples, evaluated on 400 review public test set

Metric|Value : Accuracy = 0.7650, Balanced accuracy= 0.7650

|  | Predicted 0 | Predicted 1 |
|---|---|---|
| **Actual 0** | 141 | 59 |
| **Actual 1** | 35 | 165 |

Negative reacall was 0.705, positive was 0.825, these two have matching accuarcy due to test set being balanced. They only differ only on the training folds, which is the reason for balanced accuracy.

The all positive baseline scored a 0.5 and catches 0 negatives, this only recovers 141 of 200 from a class with only 60 training examples. CV was well calcualated because balanced accuracy only had a 2.4 point gap, which mean the CV was estimated pretty good.

In [18]:
final_head = train_head(x_train, y_train)
test_probs = head_probs(final_head, x_test)
test_preds = (test_probs >= best_t).astype(int)
print("=== PUBLIC TEST ===")
_ = evaluate_model(y_test, test_preds)

=== PUBLIC TEST ===
Accuracy: 0.7650
Balanced Accuracy: 0.7650
Confusion Matrix:
[[141  59]
 [ 35 165]]


In [19]:
import os, json
os.makedirs("model_checkpoint", exist_ok=True)
torch.save(final_head.state_dict(), "model_checkpoint/head.pt")
with open("model_checkpoint/config.json", "w") as f:
    json.dump({"encoder": "distilbert-base-uncased", "hidden_dim": 768,
               "max_length": 512, "overlap": 0, "threshold": float(best_t)}, f, indent=2)
print("saved")

saved


In [20]:
with open("model_checkpoint/config.json") as f:
    cfg = json.load(f)
loaded = nn.Linear(cfg["hidden_dim"], 2)
loaded.load_state_dict(torch.load("model_checkpoint/head.pt"))
loaded.eval()
reloaded_preds = (head_probs(loaded, x_test) >= cfg["threshold"]).astype(int)
print("identical:", (reloaded_preds == test_preds).all())

identical: True


In [21]:
pd.DataFrame({"id": test["id"], "predicted_label": test_preds}).to_csv(
    "public_test_predictions.csv", index=False)
print(pd.read_csv("public_test_predictions.csv").shape)

(400, 2)


# USE OF AI

The main agaent used to assist during this project was claude. 
First used it to help conduct some research on what an optimal approch would be. It even led me to compare a few main approaches. It eventaully helped narrow it down to what I've choosen now. also had it help setup the git hub repo.

I also used it guide me through using the unfimiliar python libraries where I was able to get a lot of my tools.

After I wrote a bit of code I'd have claude critic it and attempt to improve.